# Chest X-ray — CSA/SRC pipeline (Colab)

**Run cells in order.** Each cell is small so you can re-run just one on failure without redoing the 8 GB unzip or the training.

Goal = the **§8b go/no-go gate**: train 3 models → SRC per model → cross-source collapse → SRC↔collapse coupling. GO → all 7. NO-GO → rethink SRC.

⚠️ **Stop at Cell 6 (path check) if it does not print `missing: 0`.**

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone code from GitHub

In [ ]:
import os, subprocess
REPO = 'https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    subprocess.run(['git','clone', REPO, '/content/Chest_Xray'], check=True)
os.chdir('/content/Chest_Xray')
print('cwd:', os.getcwd())
print(os.listdir('.'))

## Cell 3 — unzip data to fast local disk
Slow (~8 GB). Only needs to run **once** per session.

In [ ]:
import os, glob, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('unzipping', os.path.basename(z))
    subprocess.run(['unzip','-q','-o', z, '-d','data/raw'], check=True)
print('unzipped folders:', os.listdir('data/raw'))

## Cell 4 — bring the manifest

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE_DIR}/manifest.csv'):
    shutil.copy(f'{DRIVE_DIR}/manifest.csv', 'data/manifest.csv'); print('manifest copied from Drive')
else:
    subprocess.run(['python','scripts/build_manifest.py'], check=True); print('manifest rebuilt locally')

## Cell 5 — install deps + GPU check
`GPU:` must NOT be empty. If it is: Runtime → Change runtime type → GPU.

In [ ]:
import subprocess
subprocess.run(['pip','-q','install','-r','requirements.txt'], check=True)
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 6 — DATA PATH CHECK ⛔
**Stop here if this does not print `missing: 0`.** Paste the output to fix the folder mapping.

In [ ]:
import pandas as pd, os
df = pd.read_csv('data/manifest.csv')
missing = [p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
if missing:
    print('EXAMPLE missing path:', missing[0])
    print('actual folders under data/raw:', os.listdir('data/raw'))
    print('>>> STOP: fix folder mapping before training.')
else:
    print('>>> OK: data paths resolve. Continue.')

## Cell 7 — train 3 diverse models
Slow (~20–40 min on T4). `--resume` skips any already trained.

In [ ]:
import subprocess, sys
MODELS = ['densenet201', 'resnet50', 'vit']
EPOCHS = 20
subprocess.run([sys.executable,'scripts/train.py','--models',*MODELS,
                '--epochs',str(EPOCHS),'--batch-size','64','--resume'], check=True)

## Cell 8 — evaluate in-domain

In [ ]:
import subprocess, sys
MODELS = ['densenet201', 'resnet50', 'vit']
subprocess.run([sys.executable,'scripts/evaluate.py','--models',*MODELS], check=True)

## Cell 9 — CSA audit + SRC certificate

In [ ]:
import numpy as np, tensorflow as tf
from src.data.loaders import load_manifest, filter_rows, make_dataset, CLASS_TO_IDX
from src.shortcut import csa
from src.shortcut.src_certificate import emit_certificate
MODELS = ['densenet201', 'resnet50', 'vit']

df = load_manifest()
audit_df = filter_rows(df, split='test', roles=['in_domain'])
audit_df = audit_df.groupby('disease', group_keys=False).apply(
    lambda g: g.sample(min(len(g),300), random_state=42))
ds = make_dataset(audit_df, batch_size=64, training=False, shuffle=False)
images = np.concatenate([b[0].numpy() for b in ds], axis=0)
y_true = np.array([CLASS_TO_IDX[c] for c in audit_df['disease']])[:len(images)]

for name in MODELS:
    model = tf.keras.models.load_model(f'results/{name}/{name}_best.keras')
    audit = {ch: csa.causal_effect(model, images, y_true, ch, masks=None, n_boot=1000)
             for ch in csa.ALL_CHANNELS}
    cert = emit_certificate(name, audit, f'results/{name}/certificate.json')
    print(f'{name}: SRC={cert["src"]:.3f} valid={cert["valid"]} '
          f'dominant={max(cert["per_channel"], key=cert["per_channel"].get)}')

## Cell 10 — cross-source collapse + C3 coupling

In [ ]:
from src.shortcut import cross_domain
MODELS = ['densenet201', 'resnet50', 'vit']
cross_domain.run_cross_source_matrix(models=MODELS, batch_size=64)
cross_domain.couple_src_to_collapse()

## Cell 11 — READ THE §8b VERDICT
Decide your threshold **before** looking (no p-hacking — PAPER_OUTLINE.md §8b).

In [ ]:
import json
c = json.load(open('results/c3_coupling.json'))
print(json.dumps(c, indent=2))
if c.get('n_models',0) >= 3 and 'delta_acc' in c:
    r2 = c['delta_acc']['r2']; r = c['delta_acc']['r']
    print(f"\nSRC -> delta_acc:  r={r:+.3f}  R2={r2:.3f}")
    print('GO (proceed to all 7)' if (r > 0 and r2 >= 0.3)
          else 'WEAK/NO-GO -> rethink SRC before full runs (PAPER_OUTLINE.md §8b)')
else:
    print('Need 3 models with valid SRC + cross-source results.')

## Cell 12 — SAVE results to Drive (do not skip)
Colab `/content` is wiped at session end. This persists checkpoints + results.

In [ ]:
import subprocess, datetime
stamp = datetime.date.today().isoformat()
subprocess.run(['cp','-r','results', f'/content/drive/MyDrive/cxr_data/results_{stamp}'], check=True)
print('saved -> Drive/cxr_data/results_'+stamp)

## Cell 13 — ONLY AFTER 'GO': train remaining 4 models
Then re-run Cells 8–12 with `MODELS` = all 7 for the final C3 result.

In [ ]:
import subprocess, sys
REST = ['efficientnetb0','mobilenetv3large','xception','lenet5']
subprocess.run([sys.executable,'scripts/train.py','--models',*REST,
                '--epochs','100','--batch-size','64','--resume'], check=True)